<a href="https://colab.research.google.com/github/NancyWangareh/ReCover/blob/main/ReCover_Chatbot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReCover: Insurance Information Chatbot for Kenyan Informal Traders

This notebook builds a chatbot that answers insurance questions for market traders (for example mitumba sellers) in English, Kiswahili or Sheng. It answers **only from the approved documents** in the team repo, shows where each fact came from, and says so when the documents don't contain the answer.

## How the project requirements are met
| Requirement | Where in this notebook |
|---|---|
| Use an LLM to interpret questions in everyday language | Section 8 (`make_queries`) rewrites questions for searching; Section 9 (`extract_profile`) reads what the trader tells us |
| Retrieve relevant information from the approved documents | Sections 3-7: read the PDFs, cut them into chunks, store them in a searchable library; Section 8 (`retrieve`) searches it |
| Apply rules for eligibility, coverage, premiums, exclusions | Section 10: plain Python rules, no AI involved |
| Return source references | Section 12: every fact is labelled `[S1]` / `[R1]` and the answer ends with a source list (file + page) |
| Say when information is unavailable | `retrieve` drops weak matches; `ReCover.ask` returns a fixed message when nothing relevant is found; the instructions forbid guessing |



## 0. Install the libraries we use
- `pypdf` reads PDFs
- `langchain-text-splitters` cuts long text into chunks
- `chromadb` stores the searchable library
- `sentence-transformers` turns text into numbers (embeddings) so we can search by meaning
- `google-genai` talks to Gemini
- `pandas` + `openpyxl` read the Excel workbook

In [6]:
%pip install -q pypdf langchain-text-splitters chromadb sentence-transformers google-genai pandas openpyxl

## 1. Settings and getting the repo's data
Colab starts as an empty computer every session, so we first copy (clone) the team repo into it. All the approved documents live in the repo's `data collection` folder.

All the settings you may want to change are in one place below.

In [7]:
import json
import os
import re
import subprocess
from pathlib import Path

import pandas as pd


REPO_URL = "https://github.com/NancyWangareh/ReCover.git"
MODEL = "gemini-3.8-flash"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
COLLECTION_NAME = "recover_docs"
MIN_SIMILARITY = 0.30
TOP_K = 4
MAX_RESULTS = 5


def find_docs_dir():
    """Look for the folder holding the approved documents."""
    for candidate in [Path("ReCover/data collection"), Path("data collection")]:
        if candidate.exists():
            return candidate
    return None

if find_docs_dir() is None:
    if "<owner>" in REPO_URL:
        raise ValueError("Edit REPO_URL above and put in your team's repo link, then run this cell again.")
    subprocess.run(["git", "clone", REPO_URL], check=True)

DOCS_DIR = find_docs_dir()
if DOCS_DIR is None:
    raise FileNotFoundError("Could not find the 'data collection' folder after cloning. Check REPO_URL.")
REPO_ROOT = DOCS_DIR.parent
print("Documents folder:", DOCS_DIR)


def get_api_key():
    """Read the Gemini key from Colab Secrets, or from an environment variable when running locally.
    The key is never written in the notebook, which matters because this notebook goes on GitHub."""
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:
        key = os.environ.get("GEMINI_API_KEY")
        if not key:
            raise RuntimeError("No Gemini key found. Add GEMINI_API_KEY to Colab Secrets or set it as an environment variable.")
        return key

Documents folder: ReCover/data collection


## 2. Look at the data we have
Before building anything, list every file in the repo and check that the files this notebook relies on are present.

In [8]:

for f in sorted(REPO_ROOT.rglob("*")):
    if ".git" not in f.parts and f.is_file():
        print(f"- {f.relative_to(REPO_ROOT)} ({f.stat().st_size:,} bytes)")


needed = [
    "Microinsurance_Data_Collection_Workbook.xlsx",
    "ReCover_Microinsurance_Product_Matching_Database.xlsx",
]
print()
for name in needed:
    found = list(REPO_ROOT.rglob(name))
    print(("FOUND   " if found else "MISSING ") + name)

- .gitignore (330 bytes)
- Information_system.ipynb (543,409 bytes)
- LICENSE (1,070 bytes)
- README.md (7,992 bytes)
- data/README.md (1 bytes)
- data/ReCover_Microinsurance_Product_Matching_Database.xlsx (16,239 bytes)
- data/traders.csv (88,447 bytes)
- data collection/BRITAM-PRODUCTS-PROFILE-Final-Final.pdf (13,301,344 bytes)
- data collection/Microinsurance_Data_Collection_Workbook.xlsx (29,634 bytes)
- data collection/TCIP-BROCHURE.pdf (677,726 bytes)
- data collection/THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf (323,455 bytes)
- data collection/jinue_policy_document.pdf (312,266 bytes)
- data collection/ke-act-ln-2020-26-publication-document.pdf (84,865 bytes)
- docs/README.md (1 bytes)
- main.py (339 bytes)
- requirements.txt (75 bytes)
- src/README.md. (43 bytes)
- src/__init__.py (223 bytes)
- src/chunking.py (2,886 bytes)
- src/config.py (243 bytes)
- src/document_parser.py (2,112 bytes)
- src/embeddings.py (1,550 bytes)
- src/llm_client.py (1,322 bytes)
- src/ra

## 3. Check which PDFs the computer can actually read
Some PDFs contain real text. Others are **scans**: every page is a photo of text, and code can't read a photo. We test each PDF by asking for its text and counting the characters. Files with no text need the extra OCR step in Section 4.

In [9]:
from pypdf import PdfReader

pdf_files = sorted(DOCS_DIR.glob("*.pdf"))
needs_ocr = []

for pdf in pdf_files:
    reader = PdfReader(str(pdf))

    chars = sum(len(page.extract_text() or "") for page in reader.pages)
    print(f"{pdf.name}: {len(reader.pages)} pages, {chars:,} characters of text")
    if chars == 0:
        needs_ocr.append(pdf.name)

print("\nScanned PDFs that need OCR:", needs_ocr)

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 20 pages, 0 characters of text
TCIP-BROCHURE.pdf: 2 pages, 0 characters of text
THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf: 51 pages, 91,854 characters of text
jinue_policy_document.pdf: 18 pages, 61,684 characters of text
ke-act-ln-2020-26-publication-document.pdf: 7 pages, 16,327 characters of text

Scanned PDFs that need OCR: ['BRITAM-PRODUCTS-PROFILE-Final-Final.pdf', 'TCIP-BROCHURE.pdf']


## 4. Read the scanned PDFs with Gemini (OCR)
OCR means turning a picture of text into real text. We send each scanned PDF to Gemini and ask for a word-for-word copy, with a `=== PAGE n ===` line before each page so every fact can later be traced to its page.

Two safeguards:
- **temperature=0** asks for the most literal output (no creativity).
- **Results are saved** to `data/ocr_text/`, so re-running the notebook doesn't pay for OCR twice. Delete the `.txt` file if you want a fresh transcription.

**Always spot-check the result** (next cell). A wrong number in the transcription becomes a wrong number in the chatbot.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())

OCR_DIR = Path("data/ocr_text")
OCR_DIR.mkdir(parents=True, exist_ok=True)

OCR_PROMPT = """Transcribe ALL text in this PDF exactly as printed.
Do not summarize, translate, or correct anything, and keep every number exactly as written.
Start each page with a line like: === PAGE 1 === (number = the page's position in the PDF, starting at 1).
If a page has no text, write [no text] under its page line."""

for name in needs_ocr:
    out_file = OCR_DIR / f"{Path(name).stem}.txt"
    if out_file.exists():
        print(f"{name}: already transcribed, skipping")
        continue
    uploaded = client.files.upload(file=str(DOCS_DIR / name))
    resp = client.models.generate_content(
        model=MODEL,
        contents=[uploaded, OCR_PROMPT],
        config=types.GenerateContentConfig(temperature=0),
    )
    out_file.write_text(resp.text, encoding="utf-8")
    print(f"{name}: {len(resp.text):,} characters saved")

BRITAM-PRODUCTS-PROFILE-Final-Final.pdf: 23,979 characters saved
TCIP-BROCHURE.pdf: 6,335 characters saved


### Spot-check the transcriptions
Compare what prints here with the actual PDF pages. For the current documents these are the facts to look for:
- **Britam guide:** premium rates such as `0.175%` (fire), `0.55%` (theft), `5%` (money), `3%` (transit), `0.1%` (public liability). The All Risks rate is printed `15% (1.5%)` in the original, so a transcription that copies it as printed is correct.
- **Jubilee TCIP brochure:** the KES 15,000,000 limit, the wall and roof rule, and the excluded businesses.

If you add other scanned documents, check their key numbers the same way.

In [10]:
for name in needs_ocr:
    text = (OCR_DIR / f"{Path(name).stem}.txt").read_text(encoding="utf-8")
    print(f"\n===== {name} =====")
    for line in text.splitlines():
        if any(key in line.lower() for key in ["%", "15,000,000", "exclud", "walls"]):
            print(line)


===== BRITAM-PRODUCTS-PROFILE-Final-Final.pdf =====
• Premium rate of 0.175% (1.75%o)
• Premium rate on full cover is 0.55%
• Premium Rate on Full cover of 15%o (1.5%)
• Premium Rate is 5%
• Premium rate is 3%
• A simplified rating of 3% of the wage-roll
• Premium rate is 0.1%
The NSSF Act 2013 was introduced on 24 December 2013 to replace the NSSF Act Cap 258 of 1965. It was not fully implemented until 3 February 2023 when the Court of Appeal declared the Act constitutional and directed its implementation with immediate effect. The Act requires that both you and your employees increase your NSSF contributions from KES 200 per month to 6% of pensionable salary (subject to certain salary maximums). These contributions will be on a graduated scale referred to as tiers.
These are calculated as 6% of an employee’s pensionable income up to a maximum referred to as the Lower Earnings Limit (L.E.L). This is currently capped at KES 7,000 per month. The maximum contribution for Tier 1 contribu

## 5. Put every document into one list of pages
Each page becomes a small record with three labels: which file (`source`), which page (`page`), and its `text`. Keeping the page number is what lets the bot say "TCIP-BROCHURE.pdf, p.1" later.

- Readable PDFs: the text comes straight from `pypdf`.
- Scanned PDFs: the text comes from our saved OCR file, split at the `=== PAGE n ===` lines.

In [11]:
def split_ocr_pages(raw):
    """Cut an OCR text file into (page number, text) pairs using the === PAGE n === markers."""
    parts = re.split(r"=== PAGE (\d+) ===", raw)

    return [(int(num), text.strip()) for num, text in zip(parts[1::2], parts[2::2])]


pages = []

for pdf in pdf_files:
    if pdf.name in needs_ocr:
        raw = (OCR_DIR / f"{pdf.stem}.txt").read_text(encoding="utf-8")
        ocr_pages = split_ocr_pages(raw)
        expected = len(PdfReader(str(pdf)).pages)
        if len(ocr_pages) != expected:

            print(f"WARNING: {pdf.name} has {expected} pages but the transcription has {len(ocr_pages)}. "
                  "Delete its .txt file in data/ocr_text and re-run Section 4.")
        for num, text in ocr_pages:
            pages.append({"source": pdf.name, "page": num, "text": text})
    else:
        reader = PdfReader(str(pdf))
        for i, page in enumerate(reader.pages, start=1):
            pages.append({"source": pdf.name, "page": i, "text": (page.extract_text() or "").strip()})

print(pd.Series([p["source"] for p in pages]).value_counts().to_string())

THE_KENYA_MICROINSURANCE_POLICY_PAPER_-Final_Draft_.pdf    51
BRITAM-PRODUCTS-PROFILE-Final-Final.pdf                    20
jinue_policy_document.pdf                                  18
ke-act-ln-2020-26-publication-document.pdf                  7
TCIP-BROCHURE.pdf                                           2


## 6. Cut the pages into chunks
A whole page is too big to search well: it may cover several topics, and only one paragraph answers the question. We cut each page into pieces of about 1,000 characters.

- `chunk_overlap=150`: neighbouring chunks share 150 characters, so a sentence cut at the boundary doesn't lose its meaning.
- `separators`: the splitter prefers to cut at blank lines, then new lines, then sentence ends, and only cuts mid-word as a last resort.
- Each chunk keeps its `source` and `page` for citations.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []
for p in pages:
    if not p["text"] or p["text"] == "[no text]":
        continue
    for i, piece in enumerate(splitter.split_text(p["text"])):
        if len(piece.strip()) >= 30:
            chunks.append({
                "id": f"{p['source']}-p{p['page']}-c{i}",
                "source": p["source"],
                "page": p["page"],
                "text": piece.strip(),
            })

print(f"{len(chunks)} chunks")
print("\nExample chunk:", chunks[0]["id"])
print(chunks[0]["text"][:300])

273 chunks

Example chunk: BRITAM-PRODUCTS-PROFILE-Final-Final.pdf-p1-c0
Britam
With you every step of the way

Britam Biashara Network
Product Guide

Insurance and Investment Solutions for
business protection and growth.


## 7. Build the searchable library
An **embedding model** turns a piece of text into a list of 384 numbers. Texts with similar *meaning* get similar numbers, so "how much does theft cover cost?" lands near the paragraph about the theft premium even if the words differ.

We store these numbers in **ChromaDB**, a small database that finds the closest matches quickly. The first run downloads the embedding model, so it takes a few minutes.

In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)

vectors = embedder.encode(
    [c["text"] for c in chunks],
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
).tolist()

Path("data/chroma_db").mkdir(parents=True, exist_ok=True)
chroma = chromadb.PersistentClient(path="data/chroma_db")
if COLLECTION_NAME in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(COLLECTION_NAME)
collection = chroma.create_collection(COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    embeddings=vectors,
    metadatas=[{"source": c["source"], "page": c["page"]} for c in chunks],
)
print(f"Stored {collection.count()} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Stored 273 chunks


## 8. Finding the right pages for a question

**Why we rewrite questions first.** A trader may ask "which businesses are excluded from the Jubilee policy?" but you have two different Jubilee documents (TCIP for businesses and J-Inue for health). Searched as written, the correct page ranked 37th and never reached the chatbot. When Gemini rewrites the question with the full product names, it ranks 2nd.

**Why we also keep the original question.** When we used *only* the rewrites, the theft-premium question got worse: the long product names matched title pages better than the theft paragraph. So the original question is always searched too, and the rewrites are a bonus.

`retrieve()` runs these searches, merges duplicates, drops chunks below `MIN_SIMILARITY`, and returns the best few.

In [14]:

DOCS_GUIDE = """Documents available:
- Britam Biashara Network Product Guide: business insurance (fire, theft/burglary, money, all risks, goods in transit, work injury, public liability), plus group medical, funeral, group life
- Jubilee Traders Comprehensive Insurance Policy (TCIP) brochure: one-package business insurance for shops and traders
- Jubilee J-Inue policy document: health insurance
- Kenya Microinsurance Policy Paper (final draft)
- Kenya legal notice (file ke-act-ln-2020-26)"""

REWRITE_PROMPT = f"""You prepare search queries for a retrieval system about Kenyan insurance.
{DOCS_GUIDE}

Rewrite the trader's message as 1 to 2 standalone English search queries.
- KEEP the topic words from the message (for example 'theft', 'premium rate', 'excluded'). Do not replace them with product names.
- Add a product's full name only when the message clearly names that product, or when it is ambiguous which Jubilee or Britam product is meant.
- Never answer the question. Only write the queries.
Return a JSON list of strings."""


def make_queries(question):
    """Return the list of search queries: the trader's own question first, then up to 2 rewrites."""
    rewrites = []
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{REWRITE_PROMPT}\n\nTrader message: {question}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        parsed = json.loads(resp.text)
        if isinstance(parsed, list):
            rewrites = [q for q in parsed if isinstance(q, str)][:2]
    except Exception as e:
        print(f"(rewrite failed, using the original question only: {e})")
    return [question] + [q for q in rewrites if q != question]


def retrieve(question):
    """Find the chunks most relevant to the question. Returns [] when nothing is relevant enough."""
    best = {}

    for query in make_queries(question):
        vec = embedder.encode(query, normalize_embeddings=True).tolist()
        res = collection.query(query_embeddings=[vec], n_results=TOP_K,
                               include=["documents", "metadatas", "distances"])
        for cid, doc, meta, dist in zip(res["ids"][0], res["documents"][0],
                                        res["metadatas"][0], res["distances"][0]):
            sim = 1 - dist
            if sim >= MIN_SIMILARITY and (cid not in best or sim > best[cid]["similarity"]):
                best[cid] = {"source": meta["source"], "page": meta["page"],
                             "text": doc, "similarity": sim}

    hits = sorted(best.values(), key=lambda h: h["similarity"], reverse=True)
    return hits[:MAX_RESULTS]



for q in ["What is the premium rate for theft cover?", "Who will win the football match on Sunday?"]:
    print("\nQUESTION:", q)
    found = retrieve(q)
    if not found:
        print("  -> nothing relevant found")
    for h in found:
        print(f"  {h['source']}, p.{h['page']} ({h['similarity']:.2f})")


QUESTION: What is the premium rate for theft cover?
  TCIP-BROCHURE.pdf, p.1 (0.69)
  BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8 (0.61)
  TCIP-BROCHURE.pdf, p.1 (0.53)
  BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.1 (0.53)
  jinue_policy_document.pdf, p.13 (0.52)

QUESTION: Who will win the football match on Sunday?
  -> nothing relevant found


## 9. Read what the trader tells us
Traders give facts we need for the rules: what kind of premises they have, their stock value, their budget. Gemini reads the message and fills in a small **form**. Python then checks the form and does any arithmetic, because the AI can make mistakes with numbers.

Rules for the form:
- Anything the trader didn't clearly say stays empty (`null`). We never guess.
- The budget amount and its period (day / week / month) are kept **separate**. Python converts them later.
- The field names follow the `Feature_Mapping` sheet in `ReCover_Microinsurance_Product_Matching_Database.xlsx`.
- `clean_profile` doesn't trust Gemini's output blindly. It keeps only known fields with the right type and allowed values.

In [15]:
PROFILE_PROMPT = """You read a message from a Kenyan trader asking about insurance and extract only the facts they stated.
Messages may be English, Kiswahili or Sheng. Return a JSON object with ONLY these keys:
- stall_type: "open_air" (open-air stall, tarpaulin, no walls), "temporary_structure" (wooden or mabati kiosk), "permanent_building" (stone or brick shop, 'duka ya mawe'), or null
- business_type: short English text such as "mitumba shoes", or null
- stock_value_kes: whole number, or null
- budget_amount_kes: the amount the trader says they can pay, as a whole number, or null
- budget_period: "day", "week" or "month" (the period for that amount), or null
- wants_cover_fire, wants_cover_theft, wants_cover_health, wants_cover_accident: true if the trader says they want it or worry about it, otherwise null
- dependents: whole number, or null
Use null for anything not clearly stated. Never guess."""

STALL_TYPES = {"open_air", "temporary_structure", "permanent_building"}
PERIODS = {"day", "week", "month"}
WANTS = ["wants_cover_fire", "wants_cover_theft", "wants_cover_health", "wants_cover_accident"]


def clean_profile(raw):
    """Keep only the fields we know, with valid values. Anything else from Gemini is thrown away."""
    out = {}
    if raw.get("stall_type") in STALL_TYPES:
        out["stall_type"] = raw["stall_type"]
    if isinstance(raw.get("business_type"), str) and raw["business_type"].strip():
        out["business_type"] = raw["business_type"].strip()
    for key in ("stock_value_kes", "budget_amount_kes", "dependents"):
        v = raw.get(key)

        if isinstance(v, (int, float)) and not isinstance(v, bool) and v >= 0:
            out[key] = int(v)
    if raw.get("budget_period") in PERIODS:
        out["budget_period"] = raw["budget_period"]
    for key in WANTS:
        if raw.get(key) is True:
            out[key] = True
    return out


def extract_profile(message):
    """Ask Gemini to fill in the form from the trader's message. Returns {} if anything goes wrong."""
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=f"{PROFILE_PROMPT}\n\nTrader message: {message}",
            config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"),
        )
        parsed = json.loads(resp.text)
        return clean_profile(parsed) if isinstance(parsed, dict) else {}
    except Exception as e:
        print(f"(profile extraction failed: {e})")
        return {}


for t in ["Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?",
          "Niko na kibanda ya mitumba viatu, ni mabati, stock 300,000 bob. Naweza lipa 500 kwa mwezi.",
          "What is microinsurance?"]:
    print(t, "\n  ->", extract_profile(t), "\n")

Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali? 
  -> {'stall_type': 'permanent_building'} 

Niko na kibanda ya mitumba viatu, ni mabati, stock 300,000 bob. Naweza lipa 500 kwa mwezi. 
  -> {'stall_type': 'temporary_structure', 'business_type': 'mitumba shoes', 'stock_value_kes': 300000, 'budget_amount_kes': 500, 'budget_period': 'month'} 

What is microinsurance? 
  -> {} 



## 10. The rules (plain Python, no AI)
These rules check the trader's situation against what the documents say. Every rule result carries the file and page it comes from.

Each product gets one of three statuses:
- `may_qualify`: nothing in the documents rules the trader out, but something still has to be confirmed
- `needs_confirmation`: the documents don't say, so the insurer must confirm
- `does_not_meet`: a requirement stated in the document is not met

A status is never a plain "yes". Documents say premises "should" meet a rule, and the matching database says eligibility must be confirmed in writing.

**Where each rule comes from**
- **TCIP:** the KES 15,000,000 limit, the wall and roof materials and the excluded businesses (banks, forex bureaux, jewellery shops, manufacturing premises) are from `TCIP-BROCHURE.pdf`, p.1. The brochure gives no premium.
- **Britam Biashara:** the guide has no stall-type rule and doesn't list its exclusions; it gives premium *rates*, not a flat premium. The need to confirm open-air stall eligibility in writing is from the matching database (Products P001/P002).
- **Budget:** the conversion (weeks x 52/12, days x 7 x 52/12) follows the workbook's Affordability sheet.

These rules are written by hand from the documents, so if a document changes, the rules must be updated too.

In [16]:
TCIP_SRC = "TCIP-BROCHURE.pdf, p.1"
BRITAM_SRC = "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.7"
DB_SRC = "ReCover_Microinsurance_Product_Matching_Database.xlsx (Products P001/P002)"

TCIP_MAX_STOCK = 15_000_000
TCIP_EXCLUDED = {
    "bank": "banks",
    "forex": "forex bureaux",
    "jewel": "jewellery shops",
    "manufactur": "premises where manufacturing is carried out",
}
STALL_WORDS = {
    "open_air": "an open-air stall",
    "temporary_structure": "a temporary (wooden or mabati) kiosk",
    "permanent_building": "a permanent stone or brick shop",
}
ORDER = {"may_qualify": 0, "needs_confirmation": 1, "does_not_meet": 2}


def worse(a, b):
    """Return the more serious of two statuses, so one failed check is never overwritten by a later pass."""
    return a if ORDER[a] >= ORDER[b] else b


def monthly_budget(profile):
    """Convert the trader's budget to KES per month. Returns None if amount or period is missing."""
    amount, period = profile.get("budget_amount_kes"), profile.get("budget_period")
    if amount is None or period is None:
        return None
    factor = {"month": 1, "week": 52 / 12, "day": 7 * 52 / 12}[period]
    return round(amount * factor)


def check_tcip(profile):
    """Check the trader against the Jubilee TCIP brochure."""
    status, reasons = "may_qualify", []
    stall = profile.get("stall_type")
    stock = profile.get("stock_value_kes")
    business = (profile.get("business_type") or "").lower()


    if stock is not None and stock > TCIP_MAX_STOCK:
        status = worse(status, "does_not_meet")
        reasons.append((f"Stock of KES {stock:,} is above the KES {TCIP_MAX_STOCK:,} limit for contents and stock.", TCIP_SRC))


    for keyword, label in TCIP_EXCLUDED.items():
        if keyword in business:
            status = worse(status, "does_not_meet")
            reasons.append((f"'{profile['business_type']}' appears to match an excluded business ({label}).", TCIP_SRC))


    if stall is None:
        status = worse(status, "needs_confirmation")
        reasons.append(("Premises should have walls of reinforced concrete, concrete blocks, bricks or stone, and a hard roof of tiles, cement or metal sheets. The trader's premises type was not stated.", TCIP_SRC))
    elif stall == "permanent_building":

        reasons.append(("Stone or brick walls match the wall rule. The roof must be tiles, cement or metal sheets; the roof was not stated.", TCIP_SRC))
    else:
        status = worse(status, "does_not_meet")
        reasons.append((f"Premises should have walls of reinforced concrete, concrete blocks, bricks or stone. {STALL_WORDS[stall].capitalize()} does not match that.", TCIP_SRC))


    reasons.append(("The brochure does not state a premium.", TCIP_SRC))
    return {"product": "Jubilee Traders Comprehensive Insurance Policy (TCIP)", "status": status, "reasons": reasons}


def check_britam(profile):
    """Check the trader against the Britam Biashara guide. Always needs_confirmation, with the reason spelled out."""
    stall = profile.get("stall_type")
    reasons = [("The guide says the cover applies to all occupations other than the defined exclusions, but it does not list those exclusions or any stall-type rule.", BRITAM_SRC)]
    if stall == "open_air":
        reasons.append(("Open-air stall eligibility must be confirmed in writing by the insurer; do not assume it is covered.", DB_SRC))
    elif stall == "temporary_structure":
        reasons.append(("The guide does not say whether a temporary kiosk qualifies; the insurer must confirm in writing.", BRITAM_SRC))
    else:
        reasons.append(("Eligibility for the trader's premises must be confirmed with the insurer.", DB_SRC))
    reasons.append(("The guide gives premium rates (percentages), not a flat premium, and does not say what amount the rate applies to. A quote is needed.", "BRITAM-PRODUCTS-PROFILE-Final-Final.pdf, p.8"))
    return {"product": "Britam Biashara Cover", "status": "needs_confirmation", "reasons": reasons}


def run_rules(profile):
    """Run every product check, and build one note about the budget. Returns (results, budget_note)."""
    results = [check_tcip(profile), check_britam(profile)]
    monthly = monthly_budget(profile)
    note = None
    if monthly is not None:

        note = (f"The trader's budget is about KES {monthly:,} per month. Neither document gives a flat premium, "
                "so we cannot say whether a product fits the budget until the insurer gives a quote.")
    return results, note

### Test the rules
These checks need no internet or API key. If a rule is changed and breaks, the cell stops with an error instead of quietly giving wrong answers.

In [17]:
r, _ = run_rules({"stall_type": "permanent_building", "stock_value_kes": 300000})
assert r[0]["status"] == "may_qualify" and r[1]["status"] == "needs_confirmation"

r, _ = run_rules({"stall_type": "temporary_structure", "business_type": "mitumba shoes"})
assert r[0]["status"] == "does_not_meet"

r, note = run_rules({"stall_type": "permanent_building", "business_type": "jewellery",
                     "budget_amount_kes": 100, "budget_period": "week"})
assert r[0]["status"] == "does_not_meet"

assert run_rules({"stall_type": "permanent_building", "stock_value_kes": 20_000_000})[0][0]["status"] == "does_not_meet"
assert run_rules({"stall_type": "open_air"})[0][0]["status"] == "does_not_meet"
assert run_rules({})[0][0]["status"] == "needs_confirmation"      # nothing known -> ask, don't assume
assert monthly_budget({"budget_amount_kes": 20, "budget_period": "day"}) == 607    # matches the workbook (about 606.67)
assert monthly_budget({"budget_amount_kes": 100, "budget_period": "week"}) == 433
assert monthly_budget({}) is None
print("All rule tests passed.")
print(note)

All rule tests passed.
The trader's budget is about KES 433 per month. Neither document gives a flat premium, so we cannot say whether a product fits the budget until the insurer gives a quote.


## 11. The instructions Gemini must follow
This is the standing rulebook sent with every answer. The most important rule is **#1: use only the material we give it.** Without it, Gemini would answer from general knowledge and could invent policy details.

In [18]:
SYSTEM = """You are 'ReCover', a plain-spoken insurance information assistant for informal traders in Kenya (for example mitumba sellers).

LANGUAGE: Reply in the language the trader used: English, Kiswahili, or a Swahili/English/Sheng mix. Keep it simple, no insurance jargon.

RULES (strict):
1. Use ONLY the EXCERPTS and RULE RESULTS provided. Every number, product name, limit and exclusion must come from them.
2. After each fact, cite its label, like [S1] or [R1]. Never cite a label that was not provided.
3. If the material does not answer part of the question, say clearly that the approved documents do not specify it. Do not guess or use general knowledge.
4. Premiums in the excerpts may be percentage rates. Quote the rate as printed. Do not calculate a premium. Say the exact premium needs a quote from the insurer.
5. If an excerpt looks unclear or contradictory (for example '15% (1.5%)'), say so instead of picking one.
6. Never promise a payout. Mention the conditions and exclusions shown in the material.
7. RULE RESULTS (labelled [R1], [R2]) come from checked rules and are authoritative for eligibility.
   - may_qualify: do NOT say 'yes, you qualify'. Say they may qualify and name what still has to be confirmed.
   - needs_confirmation: say the insurer must confirm eligibility.
   - does_not_meet: say the document's requirement is not met. Do not recommend that product.
8. If a BUDGET NOTE is given, repeat it. Never say a product fits or does not fit the budget.
9. Only mention a product if it is relevant to the question."""


UNAVAILABLE = ("I don't have that information in the approved insurance documents, "
               "so I can't answer it without guessing.")

## 12. The chatbot
`ReCover` joins everything. One `ReCover()` object is one conversation, and it **remembers** what the trader has said so a follow-up like "does that fit my budget?" still knows about the kiosk.

For each message `ask()` does this:
1. Read the trader's facts into the profile (Section 9).
2. Search for relevant pages (Section 8). **If nothing relevant is found, return the fixed message and stop.**
3. Run the rules (Section 10).
4. Give Gemini the rule results and the page excerpts, each with a label (`[R1]`, `[S1]`).
5. Read back which labels the answer used and list only those as sources.

In [19]:
class ReCover:
    def __init__(self):
        self.profile = {}
        self.history = []

    def search_text(self, message):
        """Short follow-ups ('does that fit my budget?') are too vague to search alone,
        so with fewer than 8 words we attach the previous message."""
        if self.history and len(message.split()) < 8:
            return f"{self.history[-1]} {message}"
        return message

    def ask(self, message):

        self.profile.update(extract_profile(message))


        hits = retrieve(self.search_text(message))
        if not hits:
            self.history.append(message)
            return UNAVAILABLE, []


        results, budget_note = run_rules(self.profile) if self.profile else ([], None)


        labelled = {}
        rule_lines, excerpt_lines = [], []
        for i, r in enumerate(results, start=1):
            sources = list(dict.fromkeys(src for _, src in r["reasons"]))
            labelled[f"R{i}"] = f"rules check: {r['product']} (based on {'; '.join(sources)})"
            lines = "\n".join(f"  - {text} ({src})" for text, src in r["reasons"])
            rule_lines.append(f"[R{i}] {r['product']}: status = {r['status']}\n{lines}")
        for i, h in enumerate(hits, start=1):
            labelled[f"S{i}"] = f"{h['source']}, p.{h['page']}"
            excerpt_lines.append(f"[S{i}] ({h['source']}, p.{h['page']})\n{h['text']}")

        prompt = (
            f"WHAT THE TRADER HAS TOLD US SO FAR: {self.profile or 'nothing yet'}\n\n"
            f"TRADER'S EARLIER MESSAGES: {self.history[-3:] or 'none'}\n\n"
            "RULE RESULTS:\n" + ("\n\n".join(rule_lines) or "(none)") + "\n\n"
            f"BUDGET NOTE: {budget_note or '(none)'}\n\n"
            "EXCERPTS:\n" + "\n\n".join(excerpt_lines) + "\n\n"
            f"TRADER QUESTION: {message}"
        )
        resp = client.models.generate_content(
            model=MODEL, contents=prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=0.2),
        )
        text = resp.text

        cited = []
        for group in re.findall(r"\[([^\]]+)\]", text):
            for label in re.findall(r"[SR]\d+", group):
                if label not in cited:
                    cited.append(label)
        sources = [f"[{c}] {labelled[c]}" for c in cited if c in labelled]

        self.history.append(message)
        return text, sources


def chat(bot, message):
    """Send one message and print the answer with its sources."""
    print(f"Trader: {message}\n")
    text, sources = bot.ask(message)
    print(f"ReCover: {text}\n")
    if sources:
        print("Sources:\n  " + "\n  ".join(sources))
    print("\n" + "=" * 70 + "\n")

## 13. Try it
What to look for in the output:
- **Kiosk conversation:** TCIP is `does not meet`, Britam is `needs confirmation`, and the budget follow-up repeats the budget note without saying anything "fits".
- **Stone shop:** "may qualify", with the roof still to confirm, not a plain yes.
- **Football:** the fixed unavailable message, with no sources.

In [20]:
bot = ReCover()
chat(bot, "Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.")
chat(bot, "My budget ni around 500 bob a month. Je, hiyo bima inatoshea bajeti yangu?")

bot2 = ReCover()
chat(bot2, "Nina duka ya mawe Gikomba. Je, Jubilee TCIP inanikubali?")
chat(bot2, "Who will win the football match on Sunday?")

bot3 = ReCover()
chat(bot3, "What is the premium rate for theft cover?")
chat(bot3, "How much is the monthly premium for Jubilee TCIP?")

Trader: Niko na kibanda ya mitumba viatu Gikomba, ni mabati na stock yangu ni 300,000 bob. Nataka bima ya moto.

ReCover: Hujambo! Kuhusu kibanda chako cha mabati cha mitumba na mpango wa bima ya moto, hivi ndivyo ripoti zetu zinavyoonyesha:

1. **Jubilee Traders Comprehensive Insurance Policy (TCIP):**
- Bima hii haikufai kwa sababu haujatimiza masharti yake [R1]. 
- Sera ya Jubilee inahitaji kuta za duka ziwe zimejengwa kwa mawe, matofali, au saruji (concrete) [S2], [R1]. Kibanda cha mabati hakitimizi masharti haya [R1].
- Vilevile, kipeperushi chao hakitaji malipo ya premium [R1].

2. **Britam Biashara Cover (Fire & Perils Cover):**
- Bima hii inashughulikia uharibifu au hasara inayosababishwa na moto na majanga mengine yaliyotajwa [S1].
- **Kuhusu kufuzu:** Mwongozo unasema inafaa kazi zote isipokuwa zile zilizotengwa, lakini haujataja orodha ya biashara zilizotengwa wala masharti ya aina ya kibanda [R2]. Kwa sababu hiyo, Britam lazima wathibitishe wenyewe kwa maandishi kama kiband

## 14. Test with the team's own trader questions
Your workbook's `Trader_Questions` sheet lists the 21 questions the chatbot must answer. This cell runs them (each in a fresh conversation) and saves the answers to `data/test_results.csv`, so a teammate can read through and mark which ones are right.

`QUESTION_LIMIT = 5` runs only the first 5, to save time and API calls. Set it to `None` to run all 21.

**Read the answers yourself.** The chatbot passing its own tests proves nothing. A person has to check each answer against the documents, and have a native Swahili speaker review the Swahili.

In [21]:
QUESTION_LIMIT = 5


def load_trader_questions():
    """Read the Trader_Questions sheet. The real table starts a few rows down, so we find the header row by looking for 'ID'."""
    workbook = next(REPO_ROOT.rglob("Microinsurance_Data_Collection_Workbook.xlsx"), None)
    if workbook is None:
        raise FileNotFoundError("Microinsurance_Data_Collection_Workbook.xlsx was not found in the repo.")
    raw = pd.read_excel(workbook, sheet_name="Trader_Questions", header=None)
    header_row = next(i for i, row in raw.iterrows() if str(row.iloc[0]).strip() == "ID")
    df = pd.read_excel(workbook, sheet_name="Trader_Questions", header=header_row)
    df = df[df["ID"].astype(str).str.match(r"Q\d+")]
    df = df.copy()

    df["Question (English)"] = df["Question (English)"].str.replace("[amount]", "150", regex=False)
    return df


questions = load_trader_questions()
if QUESTION_LIMIT:
    questions = questions.head(QUESTION_LIMIT)

rows = []
for _, q in questions.iterrows():
    try:
        text, sources = ReCover().ask(q["Question (English)"])
    except Exception as e:
        text, sources = f"ERROR: {e}", []
    rows.append({"id": q["ID"], "category": q["Category"], "question": q["Question (English)"],
                 "answer": text, "sources": " | ".join(sources)})
    print(f"{q['ID']}: {q['Question (English)']}\n  -> {text[:200]}...\n")

results_df = pd.DataFrame(rows)
results_df.to_csv("data/test_results.csv", index=False)
print("Saved data/test_results.csv")

Q01: What is microinsurance?
  -> Based on the policy documents, here is what microinsurance is:

* **Designed for low-income households:** It is an insurance service intended to be delivered in a way that is appropriate for low-incom...

Q02: Is insurance for someone like me, who sells mitumba?
  -> Yes, insurance can cover informal and small businesses like mitumba trade, but your eligibility for specific covers needs confirmation from the insurers [R1, R2]. In Kenya, microinsurance specifically...

Q03: Is this company licensed and safe?
  -> The approved documents do not specify whether a particular company is licensed or safe, as no company name was provided and the documents do not include a list of licensed companies [S1, S3]. 

Howeve...

Q04: What happens if I get sick and cannot work?
  -> The approved documents do not specify what happens to your daily work or if you receive replacement income if you get sick and cannot work. 

Here is what the available documents do say reg

## 15. Limits of this version
- **Rules cover two business products only** (Jubilee TCIP and Britam Biashara). The matching database lists 15 products, but it has no eligibility rules or premiums for the others, so health, accident and funeral questions are answered only from retrieved text, with no rule checks.
- **No flat premiums.** Neither business document gives one, so the bot can't say whether a product fits a budget. It tells the trader to get a quote.
- **Rules are written by hand** from the documents. If a document changes, update Section 10.
- **The Britam "All Risks" rate** is printed as `15% (1.5%)` in the guide, which is unclear. The bot is told to say so.
- **The fixed "I don't have that information" message is English only.**
- **OCR can make mistakes.** Check the transcriptions of scanned PDFs (Section 4) against the originals.
- **Swahili and Sheng answers need review** by a native speaker before real traders see them.
- This tool gives information, not insurance advice. Eligibility and premiums must always be confirmed with the insurer.